In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Source Han Sans CN']  # 优先使用系统已有的中文字体

In [2]:
# 1.1 导入数据
print("=" * 50)
print("1.1 数据导入")
print("=" * 50)

df = pd.read_csv('house_price_final_train_dataset.csv')

print(f"数据形状: {df.shape}")
print(f"行数: {df.shape[0]}, 列数: {df.shape[1]}")
print("\n数据基本信息:")
print(df.info())

1.1 数据导入
数据形状: (103871, 65)
行数: 103871, 列数: 65

数据基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103871 entries, 0 to 103870
Data columns (total 65 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Price                              103871 non-null  float64
 1   lnPrice                            103871 non-null  float64
 2   decoration_精装                      103291 non-null  float64
 3   decoration_简装                      103291 non-null  float64
 4   decoration_毛坯                      103291 non-null  float64
 5   decoration_其他                      103291 non-null  float64
 6   high_dummy                         103871 non-null  float64
 7   middle_dummy                       103871 non-null  float64
 8   low_dummy                          103871 non-null  float64
 9   basement_dummy                     103871 non-null  float64
 10  top_dummy                          103871 non-nu

In [3]:
# 1.2 缺失值分析
print("=" * 50)
print("1.2 缺失值分析")
print("=" * 50)

# 计算所有列的缺失值比例
missing_analysis = pd.DataFrame({
    '缺失数量': df.isnull().sum(),
    '缺失比例(%)': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('缺失比例(%)', ascending=False)

print("所有变量的缺失值统计:")
print(missing_analysis)

# 输出缺失值较多的变量
high_missing = missing_analysis[missing_analysis['缺失比例(%)'] > 50]
if not high_missing.empty:
    print(f"\n缺失比例超过50%的变量 ({len(high_missing)}个):")
    for var in high_missing.index:
        print(f"  {var}: {high_missing.loc[var, '缺失比例(%)']}%")

1.2 缺失值分析
所有变量的缺失值统计:
                          缺失数量  缺失比例(%)
heating_central          72926    70.21
heating_self             72926    70.21
heating_fee_avg          72488    69.79
house_age_under_2_years  44510    42.85
house_age_over_2_years   44510    42.85
...                        ...      ...
surrounding_school           0     0.00
property_phone_yes           0     0.00
城市                           0     0.00
板块                           0     0.00
距市中心距离_km                    0     0.00

[65 rows x 2 columns]

缺失比例超过50%的变量 (3个):
  heating_central: 70.21%
  heating_self: 70.21%
  heating_fee_avg: 69.79%


In [4]:
# 1.3 填充subway变量的缺失值为0
print("\n" + "=" * 50)
print("填充subway变量缺失值")
print("=" * 50)

if 'subway' in df.columns:
    subway_missing_before = df['subway'].isnull().sum()
    if subway_missing_before > 0:
        df['subway'] = df['subway'].fillna(0)
        subway_missing_after = df['subway'].isnull().sum()
        print(f"subway变量: {subway_missing_before} 个缺失值已填充为0")
        print(f"填充后缺失值数量: {subway_missing_after}")
    else:
        print("subway变量没有缺失值")
else:
    print("数据集中没有subway变量")



填充subway变量缺失值
subway变量: 21236 个缺失值已填充为0
填充后缺失值数量: 0


In [5]:
# 1.4 变量填充
print("\n" + "=" * 50)
print("1.3 变量填充")
print("=" * 50)

# 定义需要填充的连续型变量
continuous_variables_to_fill = [
    'room_count', 'hall_count', '梯数', '户数', 
    'building_age', 'household_total', 'building_total', 'gas_fee_avg',
    'greening_rate', 'plot_ratio', 'property_fee_avg', 'heating_fee_avg', 'parking_spots'
]

print("开始填充连续型变量...")
for column in continuous_variables_to_fill:
    if column in df.columns:
        null_count_before = df[column].isnull().sum()
        if null_count_before > 0:
            # 第一层：按板块分组计算中位数并填充
            df[column] = df.groupby('板块')[column].transform(
                lambda x: x.fillna(x.median())
            )
            
            # 第二层：检查是否仍有缺失值，如果有则按区县分组填充
            null_count_after_plate = df[column].isnull().sum()
            if null_count_after_plate > 0:
                df[column] = df.groupby('区县')[column].transform(
                    lambda x: x.fillna(x.median())
                )
            
            # 第三层：检查是否仍有缺失值，如果有则按城市分组填充
            null_count_after_district = df[column].isnull().sum()
            if null_count_after_district > 0:
                df[column] = df.groupby('城市')[column].transform(
                    lambda x: x.fillna(x.median())
                )
            
            # 第四层：检查是否仍有缺失值，如果有则用全局中位数填充
            null_count_after_city = df[column].isnull().sum()
            if null_count_after_city > 0:
                global_median = df[column].median()
                df[column] = df[column].fillna(global_median)
            
            null_count_after = df[column].isnull().sum()
            print(f"{column}: {null_count_before} → {null_count_after} 缺失值")
        else:
            print(f"{column}: 无缺失值")
    else:
        print(f"{column}: 列不存在")

print("\n✅ 连续型变量填充完成")


1.3 变量填充
开始填充连续型变量...
room_count: 580 → 0 缺失值
hall_count: 1299 → 0 缺失值
梯数: 2619 → 0 缺失值
户数: 2619 → 0 缺失值
building_age: 35101 → 0 缺失值
household_total: 7131 → 0 缺失值
building_total: 7131 → 0 缺失值
gas_fee_avg: 32701 → 0 缺失值
greening_rate: 32883 → 0 缺失值
plot_ratio: 33154 → 0 缺失值
property_fee_avg: 31158 → 0 缺失值
heating_fee_avg: 72488 → 0 缺失值
parking_spots: 34568 → 0 缺失值

✅ 连续型变量填充完成


In [6]:
# 1.3.3 哑变量填充
print("=" * 50)
print("哑变量填充")
print("=" * 50)

# 定义需要填充的哑变量
categorical_vars_to_fill = [
    'decoration_精装', 'decoration_简装', 'decoration_毛坯', 'decoration_其他',
    'elevator_yes',
    'structure_material_brick_concrete', 'structure_material_brick_wood',
    'structure_material_frame', 'structure_material_mixed',
    'structure_material_steel', 'structure_material_steel_concrete',
    'structure_material_unknown',
    'house_age_over_2_years', 'house_age_over_5_years', 'house_age_under_2_years',
    'water_civil', 'water_commercial', 'heating_central', 'heating_self',
    'electricity_civil', 'electricity_commercial','south_dummy','north_south_dummy',
    'usage_commercial_office','usage_commercial_residential','usage_high_end_residential',
    'usage_ordinary_residential','usage_other_special','subway'
]

print("开始填充哑变量...")
for column in categorical_vars_to_fill:
    if column in df.columns:
        null_count_before = df[column].isnull().sum()
        if null_count_before > 0:
            # 第一层：按板块分组用众数填充
            df[column] = df.groupby('板块')[column].transform(
                lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)
            )
            
            # 第二层：检查是否仍有缺失值，如果有则按区县分组填充
            null_count_after_plate = df[column].isnull().sum()
            if null_count_after_plate > 0:
                df[column] = df.groupby('区县')[column].transform(
                    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)
                )
            
            # 第三层：检查是否仍有缺失值，如果有则按城市分组填充
            null_count_after_district = df[column].isnull().sum()
            if null_count_after_district > 0:
                df[column] = df.groupby('城市')[column].transform(
                    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)
                )
            
            # 第四层：检查是否仍有缺失值，如果有则用全局众数填充
            null_count_after_city = df[column].isnull().sum()
            if null_count_after_city > 0:
                global_mode = df[column].mode()[0] if not df[column].mode().empty else 0
                df[column] = df[column].fillna(global_mode)
            
            null_count_after = df[column].isnull().sum()
            print(f"{column}: {null_count_before} → {null_count_after} 缺失值")
        else:
            print(f"{column}: 无缺失值")
    else:
        print(f"{column}: 列不存在")

print("\n✅ 哑变量填充完成")

哑变量填充
开始填充哑变量...
decoration_精装: 580 → 0 缺失值
decoration_简装: 580 → 0 缺失值
decoration_毛坯: 580 → 0 缺失值
decoration_其他: 580 → 0 缺失值
elevator_yes: 12351 → 0 缺失值
structure_material_brick_concrete: 580 → 0 缺失值
structure_material_brick_wood: 580 → 0 缺失值
structure_material_frame: 580 → 0 缺失值
structure_material_mixed: 580 → 0 缺失值
structure_material_steel: 580 → 0 缺失值
structure_material_steel_concrete: 580 → 0 缺失值
structure_material_unknown: 580 → 0 缺失值
house_age_over_2_years: 44510 → 0 缺失值
house_age_over_5_years: 44510 → 0 缺失值
house_age_under_2_years: 44510 → 0 缺失值
water_civil: 30298 → 0 缺失值
water_commercial: 30298 → 0 缺失值
heating_central: 72926 → 0 缺失值
heating_self: 72926 → 0 缺失值
electricity_civil: 30292 → 0 缺失值
electricity_commercial: 30292 → 0 缺失值
south_dummy: 1 → 0 缺失值
north_south_dummy: 1 → 0 缺失值
usage_commercial_office: 1 → 0 缺失值
usage_commercial_residential: 1 → 0 缺失值
usage_high_end_residential: 1 → 0 缺失值
usage_ordinary_residential: 1 → 0 缺失值
usage_other_special: 1 → 0 缺失值
subway: 无缺失值

✅ 哑变

In [7]:
# 2. 保存至本地
# 保存到本地
output_file = 'house_price_final_train_dataset2.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')